# 🚀 Hope ML: Professional Training Pipeline

This notebook provides a robust, professional environment for training the Hope Trading Transformer. Optimized for **Antigravity IDE** and **Google Colab**.

### Key Features:
*   **IDE Integration**: Secret management via GDrive `.env` file.
*   **Path Agnostic**: Auto-detects project location in Google Drive.
*   **Observability**: Real-time TensorBoard monitoring and data distribution plots.
*   **Integrity**: Automatic repo synchronization and dependency validation.

In [ ]:
# @title ## 1. Drive Connection & Path Setup
from google.colab import drive
import os
import sys
import subprocess
from pathlib import Path

# 1. Mount Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Auto-detect Project Root
POSSIBLE_ROOTS = [
    "/content/drive/MyDrive/hope",
    "/content/drive/MyDrive/hope-trading",
    "/content/hope",
    os.getcwd()
]

PROJECT_ROOT = None
for root in POSSIBLE_ROOTS:
    if os.path.exists(os.path.join(root, "scripts/train_fixed.py")):
        PROJECT_ROOT = root
        break

if not PROJECT_ROOT:
    print("❌ Project root not found! Please ensure you have cloned the repo into your Drive.")
    # Fallback to default
    PROJECT_ROOT = "/content/drive/MyDrive/hope"
    os.makedirs(PROJECT_ROOT, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")

# 3. Add to System Path
SCRIPTS_PATH = os.path.join(PROJECT_ROOT, "scripts")
if SCRIPTS_PATH not in sys.path:
    sys.path.insert(0, SCRIPTS_PATH) # Insert at front to override any stale packages

os.chdir(PROJECT_ROOT)
print(f"✅ System path updated. Current dir: {os.getcwd()}")

In [ ]:
# @title ## 2. Repository Synchronization
REPO_URL = "https://github.com/planetazul3/hope.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

def sync_repo():
    if not os.path.exists(os.path.join(PROJECT_ROOT, ".git")):
        print(f"Cloning {REPO_URL}...")
        subprocess.check_call(f"git clone --depth 1 {REPO_URL} {PROJECT_ROOT}", shell=True)
    else:
        print(f"Updating repository ({BRANCH})...")
        subprocess.check_call(f"git -C {PROJECT_ROOT} fetch origin", shell=True)
        subprocess.check_call(f"git -C {PROJECT_ROOT} reset --hard origin/{BRANCH}", shell=True)

try:
    sync_repo()
    print("✨ Repository is up to date.")
except Exception as e:
    print(f"⚠️ Sync failed (using existing files): {e}")

In [ ]:
# @title ## 3. Hardware & Dependency Audit
import torch
import platform

print(f"💻 OS: {platform.platform()}")
print(f"🐍 Python: {platform.python_version()}")

gpu_available = torch.cuda.is_available()
if gpu_available:
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"📊 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected. Training will be extremely slow!")

print("📦 Installing/Updating optimized dependencies...")
!pip install -q torch==2.1.0 pandas==2.1.1 numpy==1.24.3 scikit-learn tqdm cryptography onnx onnxruntime tensorboard
print("✅ Dependencies verified.")

In [ ]:
# @title ## 4. Credentials & Environment Editor
# @markdown Edit your `.env` file directly from here. Values are persisted to your Google Drive.

ENV_PATH = os.path.join(PROJECT_ROOT, ".env")
EXAMPLE_PATH = os.path.join(PROJECT_ROOT, ".env.example")

# Initialize if missing
if not os.path.exists(ENV_PATH) and os.path.exists(EXAMPLE_PATH):
    subprocess.check_call(f"cp {EXAMPLE_PATH} {ENV_PATH}", shell=True)

DERIV_DEMO_TOKEN = "" # @param {type:"string"}
DERIV_REAL_TOKEN = "" # @param {type:"string"}
MODEL_SIGNING_KEY = "" # @param {type:"string"}
DERIV_SYMBOL = "R_100" # @param {type:"string"}

def manage_env():
    updates = {}
    if DERIV_DEMO_TOKEN: updates["DERIV_DEMO_TOKEN"] = DERIV_DEMO_TOKEN
    if DERIV_REAL_TOKEN: updates["DERIV_REAL_TOKEN"] = DERIV_REAL_TOKEN
    if MODEL_SIGNING_KEY: updates["MODEL_SIGNING_KEY"] = MODEL_SIGNING_KEY
    if DERIV_SYMBOL: updates["DERIV_SYMBOL"] = DERIV_SYMBOL
    
    if os.path.exists(ENV_PATH):
        with open(ENV_PATH, "r") as f: lines = f.readlines()
        new_lines = []
        keys_seen = set()
        for line in lines:
            k = line.split("=")[0].strip() if "=" in line else None
            if k in updates:
                new_lines.append(f"{k}={updates[k]}\n")
                keys_seen.add(k)
            else:
                new_lines.append(line)
        for k, v in updates.items():
            if k not in keys_seen:
                new_lines.append(f"{k}={v}\n")
        with open(ENV_PATH, "w") as f: f.writelines(new_lines)
    
    # Load into current session environment
    if os.path.exists(ENV_PATH):
        with open(ENV_PATH, "r") as f:
            for line in f:
                if "=" in line and not line.startswith("#"):
                    k, v = line.strip().split("=", 1)
                    os.environ[k] = v.strip("\"\'")
    print("✅ Environment configuration synchronized.")

manage_env()

In [ ]:
# @title ## 5. Data Integrity & Distribution Analysis
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

CSV_PATH = os.path.join(PROJECT_ROOT, "data/ticks.csv")
if os.path.exists(CSV_PATH):
    print(f"📊 Analyzing {CSV_PATH}...")
    df = pd.read_csv(CSV_PATH, header=None)
    if df.shape[1] >= 3:
        df.columns = ["symbol", "epoch", "quote"]
    else:
        df.columns = ["epoch", "quote"]
    
    print(f"✅ Total Ticks: {len(df):,}")
    
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.plot(df["quote"].tail(5000))
    plt.title("Recent Price Action (Last 5k Ticks)")
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    returns = np.log(df["quote"] / df["quote"].shift(1)).dropna()
    sns.histplot(returns, kde=True, bins=100)
    plt.title("Volatility Distribution (Log Returns)")
    plt.show()
else:
    print(f"❌ Data not found at {CSV_PATH}. Please run data collection first.")

In [ ]:
# @title ## 6. Training & Live Monitoring
%load_ext tensorboard

LOG_DIR = os.path.join(PROJECT_ROOT, "logs")
os.makedirs(LOG_DIR, exist_ok=True)

print(f"📈 Launching TensorBoard (logging to {LOG_DIR})...")
%tensorboard --logdir {LOG_DIR}

try:
    import train_fixed
    print("🏗️ Starting training engine...")
    train_fixed.main(log_dir=LOG_DIR)
except ModuleNotFoundError:
    print("❌ train_fixed not found in sys.path. Ensure you have run Cell 1!")
    print("Current sys.path:", sys.path)